# 01 · Preparación de datos (VisDrone-DET, copia de Kaggle)
Objetivo: convertir YOLO → COCO JSON, crear subconjuntos y medir la densidad de objetos.
**Decisiones a registrar en `DESIGN_LOG.md`:** subconjunto usado, clases de la copia, tratamiento de cajas inválidas.

In [ ]:
# Colab/Kaggle: clona el repo y instala dependencias (ajusta la URL)
# !git clone https://github.com/<usuario>/detr-query-saturation.git && cd detr-query-saturation
# !pip -q install -r requirements.txt
import sys; sys.path.append('../src' if __import__('os').path.exists('../src') else 'src')
from common import *
import torch, json, os, pandas as pd, matplotlib.pyplot as plt
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; SEED=42; set_seed(SEED); print(DEVICE)

In [ ]:
# Ajusta estas rutas a la copia de Kaggle (revisa la estructura con el árbol de abajo)
ROOT='/kaggle/input/<dataset-visdrone>'
for r,d,f in os.walk(ROOT):
    if r[len(ROOT):].count(os.sep)<=3: print(r, len(f),'archivos')

In [ ]:
TRAIN_IMG, TRAIN_LBL = f'{ROOT}/<train/images>', f'{ROOT}/<train/labels>'
VAL_IMG,   VAL_LBL   = f'{ROOT}/<val/images>',   f'{ROOT}/<val/labels>'
os.makedirs('../data',exist_ok=True)
print(yolo_to_coco(TRAIN_IMG,TRAIN_LBL,'../data/train.json'))
print(yolo_to_coco(VAL_IMG,VAL_LBL,'../data/val.json'))

In [ ]:
# Subconjuntos para el E2 (documenta los tamaños en el design log)
N_TRAIN, N_VAL = 600, 200
print(make_subset('../data/train.json',N_TRAIN,'../data/train_sub.json',SEED))
print(make_subset('../data/val.json',N_VAL,'../data/val_sub.json',SEED))

In [ ]:
# Densidad: objetos por imagen y techo de recall (sin modelo) para N = 100, 300, 500
cnt=objects_per_image('../data/val.json'); n=pd.Series(cnt)
print(n.describe())
for N in (100,300,500):
    print(f'N={N}: imágenes con más de N objetos = {(n>N).mean():.1%} | techo de recall = {oracle_recall(cnt,N):.1%}')
n.hist(bins=50); plt.xlabel('objetos por imagen'); plt.ylabel('imágenes'); plt.savefig('../results/hist_objetos.png',dpi=150)

**Lectura:** si el % de imágenes con más de 100 objetos es muy bajo, H1 pierde fuerza. Anótalo en el log, sea cual sea el resultado.